# camada de landing carregamento dos dados 

In [0]:
## definindo os schemas e usando uma separada para o schema da api que vai usar o bronze_schema_analytics. tambem contem a logica para a criação das pathhs 
## lembrar de nao usar til e nada pois quebra quando vamos criar problemas com a tabela ascii para dar suporte 

catalog = "medalhao_cinema"
bronze_schema_name = "bronze"
bronze_schema_analytics = "bronze_analytics" 
silver_schema_name = "silver"
gold_schema_name = "gold"
fallback = "fallback"
schema_testes = "testes"


bronze_schema = f"{catalog}.{bronze_schema_name}"
bronze_schema_api = f"{catalog}.{bronze_schema_analytics}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"
fallback_schema = f"{catalog}.{fallback}"
schema_testes = f"{catalog}.{schema_testes}"

landing_path = f"/Workspace/Users/juann.perri@gmail.com/visagio/tarefa_2/landing"

print(f"landing_path: {landing_path}")
print(f"bronze_schema: {bronze_schema}")
print(f"bronze_schema_api: {bronze_schema_api}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"fallback: {fallback_schema}")
print(f"schema_testes: {schema_testes}")


landing_path: /Workspace/Users/juann.perri@gmail.com/visagio/tarefa_2/landing
bronze_schema: medalhao_cinema.bronze
bronze_schema_api: medalhao_cinema.bronze_analytics
silver_schema: medalhao_cinema.silver
gold_schema: medalhao_cinema.gold
fallback: medalhao_cinema.fallback
schema_testes: medalhao_cinema.testes


# criar catalog e schemas 

In [0]:
# validadando a criacao das tabelas e do catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_schema_api}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_schema}")



DataFrame[]

In [0]:
# a logica de validacao das tabelas se foram importatas usando como referencia o codigo que foi mandado pelo notebook para ter como referencia 
## o codigo usa um teste com request para validar a api que vai ser usada na bronze 
import requests

expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")


url = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)")


params = {
    "@dataInicial": "'01-09-2026'",       # MM-DD-YYYY, com aspas simples
    "@dataFinalCotacao": "'01-10-2026'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
}

r = requests.get(url, params=params, timeout=30) # um bom tempo porem podemos aumentar para uma instabilidade defini uma p(x!y)
r.raise_for_status()
if r.json()["value"] == []:  # se a API estiver indisponível, pode retornar uma lista vazia
    raise Exception("A API está indisponível")
print("[OK] Teste da API") 
print(r.json()["value"][:5])

#

[OK] Todos os arquivos esperados estão na landing zone:
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)
[OK] Teste da API
[{'cotacaoCompra': 5.3701, 'dataHoraCotacao': '2026-01-09 13:09:29.054'}]


o mock com interpolacao linear nesse caso para lidar com problemas de grao proximos com pouca mudanca nesse caso podemos usar o mock porem com o metodo de linterpolacao linear, pensei em usar interpolacao dos vizinhos mais proximos para lidar porem como os valores são contralados e os outliers não são muito comuns, existem mas não muda muito fiz algo mais simples referencia 

https://medium.com/@protobioengineering/digital-signals-for-dumb-sses-part-5-how-to-fill-in-missing-data-with-interpolation-8b1dcdb9bb1b

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import functions as F

dados = r.json()["value"]
if not dados:
    raise ValueError("A API não retornou cotações.")

# Dados reais recebidos da API.
df_api = spark.createDataFrame(
    [(x["dataHoraCotacao"], float(x["cotacaoCompra"])) for x in dados],
    "instante STRING, cotacao_original DOUBLE"
).withColumn("instante", F.to_timestamp("instante"))

# Última observação de cada dia.
diario = (
    df_api.withColumn("data", F.to_date("instante"))
    .groupBy("data")
    .agg(F.max(F.struct("instante", "cotacao_original")).alias("ultimo"))
    .select("data", "ultimo.cotacao_original")
)

# Calendário diário entre a primeira e a última cotação.
calendario = diario.agg(
    F.min("data").alias("inicio"),
    F.max("data").alias("fim")
).select(F.explode(F.sequence("inicio", "fim")).alias("data"))

# Pandas API on Spark: interpolação linear.
df = (
    calendario.join(diario, "data", "left")
    .pandas_api(index_col="data")
    .sort_index()
)

df["cotacao_preenchida"] = df["cotacao_original"].interpolate(
    method="linear" # podemos testar outros metodos de interpolacao
)
df["eh_interpolado"] = df["cotacao_original"].isna()

df_resultado = (
    df.reset_index().to_spark()
    .withColumn("cotacao_preenchida", F.round("cotacao_preenchida", 4))
    .withColumn("origem_base", F.lit("API_BCB"))
    .withColumn("uso", F.lit("TESTE"))
)

# Persistir para uso posterior nos testes de fallback.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.testes")

df_resultado.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.testes.tb_cotacao_fallback"
)

display(df_resultado.orderBy("data"))
dados = r.json()["value"]
if not dados:
    raise ValueError("A API não retornou cotações.")

# Dados reais recebidos da API.
df_api = spark.createDataFrame(
    [(x["dataHoraCotacao"], float(x["cotacaoCompra"])) for x in dados],
    "instante STRING, cotacao_original DOUBLE"
).withColumn("instante", F.to_timestamp("instante"))

# Última observação de cada dia.
diario = (
    df_api.withColumn("data", F.to_date("instante"))
    .groupBy("data")
    .agg(F.max(F.struct("instante", "cotacao_original")).alias("ultimo"))
    .select("data", "ultimo.cotacao_original")
)

# Calendário diário entre a primeira e a última cotação.
calendario = diario.agg(
    F.min("data").alias("inicio"),
    F.max("data").alias("fim")
).select(F.explode(F.sequence("inicio", "fim")).alias("data"))

# Pandas API on Spark: interpolação linear.
df = (
    calendario.join(diario, "data", "left")
    .pandas_api(index_col="data")
    .sort_index()
)

df["cotacao_preenchida"] = df["cotacao_original"].interpolate(
    method="linear"
)
df["eh_interpolado"] = df["cotacao_original"].isna()

df_resultado = (
    df.reset_index().to_spark()
    .withColumn("cotacao_preenchida", F.round("cotacao_preenchida", 4))
    .withColumn("origem_base", F.lit("API_BCB"))
    .withColumn("uso", F.lit("TESTE"))
)

# Persistir para uso posterior nos testes de fallback.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.testes")

df_resultado.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.testes.tb_cotacao_fallback"
)

display(df_resultado.orderBy("data"))

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1070: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


data,cotacao_original,cotacao_preenchida,eh_interpolado,origem_base,uso
2026-01-09,5.3701,5.3701,false,API_BCB,TESTE


fallback - usando 